In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from matplotlib.lines import Line2D
from matplotlib.legend import Legend

Line2D._us_dashSeq = property(lambda self: self._dash_pattern[1])
Line2D._us_dashOffset = property(lambda self: self._dash_pattern[0])
Legend._ncol = property(lambda self: self._ncols)

plot_scale = 1.0

In [3]:
from utils.datasets.base_dataset import QUERY_TYPE
from pathlib import Path
import pandas as pd
from utils.helpers import to_pow_10
from utils.helpers import pretty_print_counts
from utils.format import map_df_readable


In [4]:
results_dir = Path("scratch/results")
figures_dir = Path("scratch/figures")

all_timings = pd.read_csv(results_dir / "sub_bsbm_timings.csv")


In [5]:
all_timings["query_id"].unique()

array([-1,  0])

In [6]:
all_timings

,query_id,timing_key,timings,dataset,size,estimated_size,engine,query_type,difficulty
0,-1,processQueryAndSendResult,5.468080e+05,bsbm_1,5595,10,QLever,embedded,easy
1,0,readWordFromDisk,1.623000e+03,bsbm_1,5595,10,QLever,embedded,easy
2,0,readWordFromDisk,4.110000e+02,bsbm_1,5595,10,QLever,embedded,easy
3,0,readWordFromDisk,4.110000e+02,bsbm_1,5595,10,QLever,embedded,easy
4,0,readWordFromDisk,3.110000e+02,bsbm_1,5595,10,QLever,embedded,easy
...,...,...,...,...,...,...,...,...,...
1062568,0,tensorIndexLookup,2.087000e+04,bsbm_3,383751,1000,QLever (no Tensor Vocabulary),index,hard
1062569,0,tensorIndexLookup,2.088000e+04,bsbm_3,383751,1000,QLever (no Tensor Vocabulary),index,hard
1062570,0,tensorIndexComputeResult,1.314641e+09,bsbm_3,383751,1000,QLever (no Tensor Vocabulary),index,hard
1062571,0,processQueryAndSendResult,1.319387e+09,bsbm_3,383751,1000,QLever (no Tensor Vocabulary),index,hard


In [7]:
all_timings_sums = (
    all_timings.groupby(
        [
            "query_id",
            "timing_key",
            "engine",
            "query_type",
            "difficulty",
            "dataset",
            "size",
            "estimated_size",
        ]
    )[["timings"]]
    .agg(["sum", "count"])
    .reset_index()
    .sort_values(by=["query_id", "timing_key"])
)
all_timings_sums

query_id                 timing_key                         engine  \
                                                                         
0         -1  processQueryAndSendResult                         QLever   
1         -1  processQueryAndSendResult                         QLever   
2         -1  processQueryAndSendResult                         QLever   
3         -1  processQueryAndSendResult                         QLever   
4         -1  processQueryAndSendResult                         QLever   
..       ...                        ...                            ...   
168        0             totalExecution  QLever (no Tensor Vocabulary)   
169        0             totalExecution  QLever (no Tensor Vocabulary)   
170        0             totalExecution  QLever (no Tensor Vocabulary)   
171        0             totalExecution  QLever (no Tensor Vocabulary)   
172        0             totalExecution  QLever (no Tensor Vocabulary)   

    query_type difficulty dataset    size estimated_size       timings        
                                                                   sum count  
0     embedded       easy  bsbm_1    5595             10  5.468080e+05     1  
1     embedded       easy  bsbm_2   42417            100  1.010225e+06     1  
2     embedded       easy  bsbm_3  383751           1000  1.050963e+06     1  
3     embedded       hard  bsbm_1    5595             10  7.814570e+05     1  
4        index       easy  bsbm_1    5595             10  6.738310e+05     1  
..         ...        ...     ...     ...            ...           ...   ...  
168      index       easy  bsbm_2   42417            100  1.708221e+07     1  
169      index       easy  bsbm_3  383751           1000  4.117537e+07     1  
170      index       hard  bsbm_1    5595             10  3.145194e+07     1  
171      index       hard  bsbm_2   42417            100  1.316302e+08     1  
172      index       hard  bsbm_3  383751           1000  1.328818e+09     1  

[173 rows x 10 columns]

In [8]:

all_timings_mapped = map_df_readable(all_timings_sums).reset_index()
all_timings_mapped

index query_id                 timing_key            engine query_type  \
                                                                             
0       0       -1  processQueryAndSendResult    \systemname-TV   Embedded   
1       1       -1  processQueryAndSendResult    \systemname-TV   Embedded   
2       2       -1  processQueryAndSendResult    \systemname-TV   Embedded   
3       3       -1  processQueryAndSendResult    \systemname-TV   Embedded   
4       4       -1  processQueryAndSendResult    \systemname-TV      Index   
..    ...      ...                        ...               ...        ...   
168   168        0             totalExecution  \systemname-Base      Index   
169   169        0             totalExecution  \systemname-Base      Index   
170   170        0             totalExecution  \systemname-Base      Index   
171   171        0             totalExecution  \systemname-Base      Index   
172   172        0             totalExecution  \systemname-Base      Index   

    difficulty dataset    size estimated_size       timings        
                                                        sum count  
0         Scan  bsbm_1    5595             10  5.468080e+05     1  
1         Scan  bsbm_2   42417            100  1.010225e+06     1  
2         Scan  bsbm_3  383751           1000  1.050963e+06     1  
3         Join  bsbm_1    5595             10  7.814570e+05     1  
4         Scan  bsbm_1    5595             10  6.738310e+05     1  
..         ...     ...     ...            ...           ...   ...  
168       Scan  bsbm_2   42417            100  1.708221e+07     1  
169       Scan  bsbm_3  383751           1000  4.117537e+07     1  
170       Join  bsbm_1    5595             10  3.145194e+07     1  
171       Join  bsbm_2   42417            100  1.316302e+08     1  
172       Join  bsbm_3  383751           1000  1.328818e+09     1  

[173 rows x 11 columns]

In [9]:
all_timings_mapped["timing_key"].unique(), all_timings_mapped["engine"].unique()

(<ArrowStringArray>
 ['processQueryAndSendResult',         'idToStringAndType',
             'readTensorData',          'readWordFromDisk',
     'tensorCosineSimilarity',          'tensorFromBuffer',
           'tensorFromString',  'tensorIndexComputeResult',
          'tensorIndexLookup',            'totalExecution']
 Length: 10, dtype: str,
 <ArrowStringArray>
 ['\systemname-TV', '\systemname-Base', 'RDFTensor']
 Length: 3, dtype: str)

In [10]:
all_timings_mapped[
    (all_timings_mapped["engine"] == "\systemname-TV") & (all_timings_mapped["query_type"] == "Index")
]

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3836584/2674607361.py:2: SyntaxWarning: invalid escape sequence '\s'
  (all_timings_mapped["engine"] == "\systemname-TV") & (all_timings_mapped["query_type"] == "Index")


index query_id                 timing_key          engine query_type  \
                                                                           
4       4       -1  processQueryAndSendResult  \systemname-TV      Index   
5       5       -1  processQueryAndSendResult  \systemname-TV      Index   
6       6       -1  processQueryAndSendResult  \systemname-TV      Index   
7       7       -1  processQueryAndSendResult  \systemname-TV      Index   
8       8       -1  processQueryAndSendResult  \systemname-TV      Index   
9       9       -1  processQueryAndSendResult  \systemname-TV      Index   
20     20        0          idToStringAndType  \systemname-TV      Index   
21     21        0          idToStringAndType  \systemname-TV      Index   
22     22        0          idToStringAndType  \systemname-TV      Index   
37     37        0  processQueryAndSendResult  \systemname-TV      Index   
38     38        0  processQueryAndSendResult  \systemname-TV      Index   
39     39        0  processQueryAndSendResult  \systemname-TV      Index   
40     40        0  processQueryAndSendResult  \systemname-TV      Index   
41     41        0  processQueryAndSendResult  \systemname-TV      Index   
42     42        0  processQueryAndSendResult  \systemname-TV      Index   
57     57        0             readTensorData  \systemname-TV      Index   
58     58        0             readTensorData  \systemname-TV      Index   
59     59        0             readTensorData  \systemname-TV      Index   
60     60        0             readTensorData  \systemname-TV      Index   
61     61        0             readTensorData  \systemname-TV      Index   
62     62        0             readTensorData  \systemname-TV      Index   
67     67        0           readWordFromDisk  \systemname-TV      Index   
68     68        0           readWordFromDisk  \systemname-TV      Index   
69     69        0           readWordFromDisk  \systemname-TV      Index   
70     70        0           readWordFromDisk  \systemname-TV      Index   
71     71        0           readWordFromDisk  \systemname-TV      Index   
72     72        0           readWordFromDisk  \systemname-TV      Index   
99     99        0           tensorFromBuffer  \systemname-TV      Index   
100   100        0           tensorFromBuffer  \systemname-TV      Index   
101   101        0           tensorFromBuffer  \systemname-TV      Index   
102   102        0           tensorFromBuffer  \systemname-TV      Index   
103   103        0           tensorFromBuffer  \systemname-TV      Index   
104   104        0           tensorFromBuffer  \systemname-TV      Index   
112   112        0           tensorFromString  \systemname-TV      Index   
113   113        0           tensorFromString  \systemname-TV      Index   
114   114        0           tensorFromString  \systemname-TV      Index   
125   125        0   tensorIndexComputeResult  \systemname-TV      Index   
126   126        0   tensorIndexComputeResult  \systemname-TV      Index   
127   127        0   tensorIndexComputeResult  \systemname-TV      Index   
128   128        0   tensorIndexComputeResult  \systemname-TV      Index   
129   129        0   tensorIndexComputeResult  \systemname-TV      Index   
130   130        0   tensorIndexComputeResult  \systemname-TV      Index   
137   137        0          tensorIndexLookup  \systemname-TV      Index   
138   138        0          tensorIndexLookup  \systemname-TV      Index   
139   139        0          tensorIndexLookup  \systemname-TV      Index   
140   140        0          tensorIndexLookup  \systemname-TV      Index   
141   141        0          tensorIndexLookup  \systemname-TV      Index   
142   142        0          tensorIndexLookup  \systemname-TV      Index   
157   157        0             totalExecution  \systemname-TV      Index   
158   158        0             totalExecution  \systemname-TV      Index   
159   159        0             totalExecution  \syst

In [11]:
all_timings_mapped[
    (all_timings_mapped["engine"] == "RDFTensor") & (all_timings_mapped["query_type"] == "Embedded")
]

index query_id              timing_key     engine query_type difficulty  \
                                                                              
83     83        0  tensorCosineSimilarity  RDFTensor   Embedded       Scan   
84     84        0  tensorCosineSimilarity  RDFTensor   Embedded       Scan   
85     85        0  tensorCosineSimilarity  RDFTensor   Embedded       Scan   
86     86        0  tensorCosineSimilarity  RDFTensor   Embedded       Join   
105   105        0        tensorFromString  RDFTensor   Embedded       Scan   
106   106        0        tensorFromString  RDFTensor   Embedded       Scan   
107   107        0        tensorFromString  RDFTensor   Embedded       Scan   
108   108        0        tensorFromString  RDFTensor   Embedded       Join   
149   149        0          totalExecution  RDFTensor   Embedded       Scan   
150   150        0          totalExecution  RDFTensor   Embedded       Scan   
151   151        0          totalExecution  RDFTensor   Embedded       Scan   
152   152        0          totalExecution  RDFTensor   Embedded       Join   

    dataset    size estimated_size       timings         
                                             sum  count  
83   bsbm_1    5595             10  3.371855e+07    213  
84   bsbm_2   42417            100  1.010415e+08   2375  
85   bsbm_3  383751           1000  4.200944e+08  20548  
86   bsbm_1    5595             10  6.679045e+08  45369  
105  bsbm_1    5595             10  1.824987e+08     11  
106  bsbm_2   42417            100  2.363508e+08    101  
107  bsbm_3  383751           1000  4.530973e+08   1000  
108  bsbm_1    5595             10  2.594637e+08    144  
149  bsbm_1    5595             10  2.801952e+08      1  
150  bsbm_2   42417            100  4.602697e+08      1  
151  bsbm_3  383751           1000  1.149482e+09      1  
152  bsbm_1    5595             10  1.529738e+09      1

In [12]:
def map_group_stats(df: pd.DataFrame, value_col: str = "timings"):
    stat_mapping = {
        "totalExecution": "Total",
        "readTensorData": "TV Read",
        "idToStringAndType": "RDF Read",
        "tensorCosineSimilarity": "Dot Product",
        "tensorFromBuffer": "TV Parse",
        "tensorFromString": "JSON Parse",
        "tensorIndexComputeResult": "Index Creation",
        "tensorIndexLookup": "Index Lookup",
        # "processQueryAndSendResult": "Query Processing",
    }
    df_cp = df.copy()
    df_cp["timing_key"] = df_cp["timing_key"].map(stat_mapping)

    df_cp = df_cp[df_cp["timing_key"].notna()]

    # timing mappings to columns

    df_grouped = (
        df_cp.groupby(
            [
                "engine",
                "query_type",
                "timing_key",
                "difficulty",
                # "dataset",
                "estimated_size",
            ]
        )[[("timings", "sum"), ("timings", "count")]]
        .median()
        .reset_index()
        .sort_values(by=["engine", "query_type", "timing_key"])
    )
    df_grouped[("timings", "sum")] = (
        df_grouped[("timings", "sum")] / 1e6
    )  # convert to ms
    return df_grouped


all_timings_grouped = map_group_stats(all_timings_mapped)
all_timings_grouped


engine query_type   timing_key difficulty estimated_size  \
                                                                        
0         RDFTensor   Embedded  Dot Product       Join             10   
1         RDFTensor   Embedded  Dot Product       Scan             10   
2         RDFTensor   Embedded  Dot Product       Scan            100   
3         RDFTensor   Embedded  Dot Product       Scan           1000   
4         RDFTensor   Embedded   JSON Parse       Join             10   
..              ...        ...          ...        ...            ...   
108  \systemname-TV      Index        Total       Join            100   
109  \systemname-TV      Index        Total       Join           1000   
110  \systemname-TV      Index        Total       Scan             10   
111  \systemname-TV      Index        Total       Scan            100   
112  \systemname-TV      Index        Total       Scan           1000   

        timings           
            sum    count  
0    667.904489  45369.0  
1     33.718549    213.0  
2    101.041487   2375.0  
3    420.094354  20548.0  
4    259.463661    144.0  
..          ...      ...  
108   68.249702      1.0  
109  795.182943      1.0  
110   22.271156      1.0  
111   17.388582      1.0  
112   28.055429      1.0  

[113 rows x 7 columns]

In [13]:
def multileve_timings_to_str(df: pd.DataFrame, col="timings"):
    df_cp = df.copy()

    def combine_multi_level(row):
        # print(row)
        summed =f"{row['sum']:.2f}" if not pd.isna(row["sum"]) else "-"
        counts = int(row["count"]) if not pd.isna(row["count"]) else "-"
        return f"{summed} ({counts})"

    df_cp["combined"] = df_cp[col].apply(combine_multi_level, axis=1)
    return df_cp


all_timings_grouped_combined = multileve_timings_to_str(all_timings_grouped).drop(columns=[("timings", "sum"), ("timings", "count")]).droplevel(1, axis=1)
all_timings_grouped_combined

,engine,query_type,timing_key,difficulty,estimated_size,combined
0,RDFTensor,Embedded,Dot Product,Join,10,667.90 (45369)
1,RDFTensor,Embedded,Dot Product,Scan,10,33.72 (213)
2,RDFTensor,Embedded,Dot Product,Scan,100,101.04 (2375)
3,RDFTensor,Embedded,Dot Product,Scan,1000,420.09 (20548)
4,RDFTensor,Embedded,JSON Parse,Join,10,259.46 (144)
...,...,...,...,...,...,...
108,\systemname-TV,Index,Total,Join,100,68.25 (1)
109,\systemname-TV,Index,Total,Join,1000,795.18 (1)
110,\systemname-TV,Index,Total,Scan,10,22.27 (1)
111,\systemname-TV,Index,Total,Scan,100,17.39 (1)


In [14]:
all_timings_grouped_combined["estimated_size_p10"] = all_timings_grouped_combined["estimated_size"].apply(to_pow_10)

In [15]:
all_timings_pivot = (
    all_timings_grouped_combined.pivot_table(
        index=["difficulty", "estimated_size_p10", "engine", "query_type"],
        columns=["timing_key"],
        values="combined",
        aggfunc=lambda x: " | ".join(x),
    )
    .reset_index()
    .fillna("-")
    .set_index(["difficulty", "estimated_size_p10", "engine", "query_type"])
)
all_timings_pivot

timing_key                                                    Dot Product  \
difficulty estimated_size_p10 engine           query_type                   
Join       $10$               RDFTensor        Embedded    667.90 (45369)   
                              \systemname-Base Embedded      5.96 (45369)   
                                               Index                    -   
                              \systemname-TV   Embedded      5.84 (45369)   
                                               Index                    -   
           $10^{2}$           \systemname-Base Index                    -   
                              \systemname-TV   Index                    -   
           $10^{3}$           \systemname-Base Index                    -   
                              \systemname-TV   Index                    -   
Scan       $10$               RDFTensor        Embedded       33.72 (213)   
                              \systemname-Base Embedded        0.04 (213)   
                                               Index                    -   
                              \systemname-TV   Embedded        0.04 (213)   
                                               Index                    -   
           $10^{2}$           RDFTensor        Embedded     101.04 (2375)   
                              \systemname-Base Embedded       0.31 (2375)   
                                               Index                    -   
                              \systemname-TV   Embedded       0.31 (2375)   
                                               Index                    -   
           $10^{3}$           RDFTensor        Embedded    420.09 (20548)   
                              \systemname-Base Embedded      2.52 (20548)   
                                               Index                    -   
                              \systemname-TV   Embedded      2.64 (20548)   
                                               Index                    -   

timing_key                                                Index Creation  \
difficulty estimated_size_p10 engine           query_type                  
Join       $10$               RDFTensor        Embedded                -   
                              \systemname-Base Embedded                -   
                                               Index           16.54 (1)   
                              \systemname-TV   Embedded                -   
                                               Index           11.63 (1)   
           $10^{2}$           \systemname-Base Index          121.82 (1)   
                              \systemname-TV   Index           57.31 (1)   
           $10^{3}$           \systemname-Base Index         1314.64 (1)   
                              \systemname-TV   Index          781.11 (1)   
Scan       $10$               RDFTensor        Embedded                -   
                              \systemname-Base Embedded                -   
                                               Index            8.03 (1)   
                              \systemname-TV   Embedded                -   
                                               Index           17.49 (1)   
           $10^{2}$           RDFTensor        Embedded                -   
                              \systemname-Base Embedded                -   
                                               Index           10.49 (1)   
                              \systemname-TV   Embedded                -   
                                               Index           10.56 (1)   
           $10^{3}$           RDFTensor        Embedded                -   
                              \systemname-Base Embedded                -   
                                               Index           32.68 (1)   
                              \systemname-TV   Embedded                -   
                                               Index           19.14 (1)   

timing_key          

In [16]:
all_timings_pivot.columns.name = None
all_timings_pivot.index.rename(["Difficulty", "$n$", "Engine", "Query Type"], inplace=True)
all_timings_pivot

Dot Product  \
Difficulty $n$      Engine           Query Type                   
Join       $10$     RDFTensor        Embedded    667.90 (45369)   
                    \systemname-Base Embedded      5.96 (45369)   
                                     Index                    -   
                    \systemname-TV   Embedded      5.84 (45369)   
                                     Index                    -   
           $10^{2}$ \systemname-Base Index                    -   
                    \systemname-TV   Index                    -   
           $10^{3}$ \systemname-Base Index                    -   
                    \systemname-TV   Index                    -   
Scan       $10$     RDFTensor        Embedded       33.72 (213)   
                    \systemname-Base Embedded        0.04 (213)   
                                     Index                    -   
                    \systemname-TV   Embedded        0.04 (213)   
                                     Index                    -   
           $10^{2}$ RDFTensor        Embedded     101.04 (2375)   
                    \systemname-Base Embedded       0.31 (2375)   
                                     Index                    -   
                    \systemname-TV   Embedded       0.31 (2375)   
                                     Index                    -   
           $10^{3}$ RDFTensor        Embedded    420.09 (20548)   
                    \systemname-Base Embedded      2.52 (20548)   
                                     Index                    -   
                    \systemname-TV   Embedded      2.64 (20548)   
                                     Index                    -   

                                                Index Creation  \
Difficulty $n$      Engine           Query Type                  
Join       $10$     RDFTensor        Embedded                -   
                    \systemname-Base Embedded                -   
                                     Index           16.54 (1)   
                    \systemname-TV   Embedded                -   
                                     Index           11.63 (1)   
           $10^{2}$ \systemname-Base Index          121.82 (1)   
                    \systemname-TV   Index           57.31 (1)   
           $10^{3}$ \systemname-Base Index         1314.64 (1)   
                    \systemname-TV   Index          781.11 (1)   
Scan       $10$     RDFTensor        Embedded                -   
                    \systemname-Base Embedded                -   
                                     Index            8.03 (1)   
                    \systemname-TV   Embedded                -   
                                     Index           17.49 (1)   
           $10^{2}$ RDFTensor        Embedded                -   
                    \systemname-Base Embedded                -   
                                     Index           10.49 (1)   
                    \systemname-TV   Embedded                -   
                                     Index           10.56 (1)   
           $10^{3}$ RDFTensor        Embedded                -   
                    \systemname-Base Embedded                -   
                                     Index           32.68 (1)   
                    \systemname-TV   Embedded                -   
                                     Index           19.14 (1)   

                                                   Index Lookup  \
Difficulty $n$      Engine           Query Type                   
Join       $10$     RDFTensor        Embedded                 -   
                    \systemname-Base Embedded                 -   
                                     Index           0.93 (213)   
                    \systemname-TV   Embedded                 -   
                                     Index           1.20 (213)   
           $10^{2}$ \systemname-Base Index         22.29 (2375)   
                    \systemname-TV   Index         22.03

In [17]:
table_dir = figures_dir / "tables" / "subtimings"
table_dir.mkdir(parents=True, exist_ok=True)

In [18]:
from utils.helpers import collapse_first_rows, to_multicol, add_args


out_path = table_dir / "bsbm_subtimings.tex"
with open(out_path, "w") as out_f:
    all_timings_pivot.style.format().to_latex(
        buf=out_f,
        caption="Sub-Timing for the \\bsbmname dataset. The timing are for a cold-start query after suming over the timing hits, with the number hits $h$ in parentheses.",
        label="tab:bsbm_subtimings",
        clines="all;data",
        hrules=True,

    )
    latex_str = ""
collapse_first_rows(out_path)
to_multicol(out_path)
add_args(out_path, before_caption="\\centering", after_caption="\\tiny")

In [19]:
for difficulty in all_timings_grouped_combined["difficulty"].unique():
    for size in all_timings_grouped_combined["estimated_size"].unique():
        subset: pd.DataFrame = all_timings_grouped_combined[
            (all_timings_grouped_combined["difficulty"] == difficulty)
            & (all_timings_grouped_combined["estimated_size"] == size)
        ]
        subset_pivot = (
            subset.pivot_table(
                index=["engine", "query_type"],
                columns=["timing_key"],
                values="combined",
                aggfunc=lambda x: " | ".join(x),
            )
            .reset_index()
            .fillna("-")
            .set_index(["engine", "query_type"])
        )
        subset_pivot.columns.name = None
        subset_pivot.index.rename(["Engine", "Query Type"], inplace=True)
        out_path = table_dir / f"bsbm_subtimings_{difficulty}_{size}.tex"
        with open(out_path, "w") as out_f:
            subset_pivot.style.format().to_latex(
                buf=out_f,
                caption=f'Sub-Timing for the \\bsbmname dataset with query "{difficulty}" and $m=${to_pow_10(size)}. The timing are for a cold-start query after suming over the timing hits, with the number hits $h$ in parentheses.',
                label=f"tab:bsbm_subtimings_{difficulty}_{size}",
                clines="all;data",
                hrules=True,
            )

        collapse_first_rows(out_path)
        to_multicol(out_path)
        add_args(out_path, before_caption="\\centering", after_caption="\\tiny")